In [ ]:
# ============================================================
# CELL 1 — SETUP & DRIVE MOUNT
# ============================================================
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

base_dir = '/content/drive/MyDrive/MSFin_Python'
data_dir = os.path.join(base_dir, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Environment ready: {data_dir}')

In [ ]:
# ============================================================
# CELL 2 — DATASET REGISTRY
# ============================================================
# Each entry defines exactly how that dataset should be handled.
# 'mappings'   : raw string values -> numeric (target column or categoricals)
# 'dummies'    : columns to one-hot encode (converted to str first to absorb NaN/None)
# 'continuous' : columns to z-score standardize
# 'target'     : binary outcome column name (post-mapping)

DATASET_REGISTRY = {
    'UCLA': {
        'file':       'LR_UCLA.csv',
        'target':     'admit',
        'mappings':   {},
        'dummies':    ['rank'],
        'continuous': ['gre', 'gpa']
    },
    'PIMA': {
        'file':       'LR_PIMA.csv',
        'target':     'outcome',
        'mappings':   {},
        'dummies':    [],
        'continuous': ['glucose', 'bloodpressure', 'bmi', 'age', 'insulin']
    },
    'Titanic': {
        'file':       'LR_Titanic200.csv',
        'target':     'survived',          # lowercased by loader
        'mappings':   {},
        'dummies':    ['sex', 'pclass', 'embarked'],   # lowercased by loader
        'continuous': ['age', 'fare', 'sibsp', 'parch']
    }
}

print('Registry loaded:', list(DATASET_REGISTRY.keys()))

In [ ]:
# ============================================================
# CELL 3 — UNIVERSAL LOADER & PREFLIGHT CHECK
# ============================================================

def universal_loader(dataset_name):
    """
    Loads, cleans, and prepares (y, X) for statsmodels Logit.
    Order of operations matters — do not reorder without reason.
    """
    recipe = DATASET_REGISTRY[dataset_name]
    path   = os.path.join(data_dir, recipe['file'])

    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing: {path}')

    df = pd.read_csv(path)

    # --- Step 1: Normalize column names ---
    # Lowercase + strip so 'Survived', ' survived ', 'SURVIVED' all resolve.
    df.columns = [c.lower().strip() for c in df.columns]

    # --- Step 2: Apply explicit value mappings (e.g. 1->1, 2->0) ---
    if recipe.get('mappings'):
        for col, mapping in recipe['mappings'].items():
            col_l = col.lower().strip()
            if col_l in df.columns:
                df[col_l] = df[col_l].map(mapping)

    # --- Step 3: One-hot encode categoricals ---
    # Force to string FIRST so NaN becomes the literal string 'nan',
    # which get_dummies treats as a valid category rather than dropping.
    # drop_first=True avoids perfect multicollinearity.
    for col in recipe['dummies']:
        col_l = col.lower().strip()
        if col_l in df.columns:
            df[col_l] = df[col_l].astype(str)
    dummy_cols = [c.lower().strip() for c in recipe['dummies'] if c.lower().strip() in df.columns]
    if dummy_cols:
        df = pd.get_dummies(df, columns=dummy_cols, drop_first=True)

    # --- Step 4: Standardize continuous features (z-score) ---
    # Coerce to numeric first; replace non-numeric with column median.
    for col in recipe['continuous']:
        col_l = col.lower().strip()
        if col_l in df.columns:
            s = pd.to_numeric(df[col_l], errors='coerce')
            s = s.fillna(s.median())
            df[col_l] = (s - s.mean()) / s.std()

    # --- Step 5: Final numeric enforcement ---
    # Any columns still non-numeric (e.g. bool dummies) get cast.
    # Median-fill remaining NaNs in numeric columns; zero-fill everything else.
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df = df.fillna(0)

    # --- Step 6: Separate target and features ---
    target = recipe['target'].lower().strip()
    if target not in df.columns:
        raise KeyError(f"Target column '{target}' not found. Columns: {list(df.columns)}")

    y = df[target].astype(int)
    X = df.drop(columns=[target])

    # --- Step 7: Drop zero-variance columns (causes singular matrix in Logit) ---
    varying = X.columns[X.nunique() > 1]
    X = X[varying]

    # --- Step 8: Add statsmodels intercept ---
    X = sm.add_constant(X, has_constant='add')

    return y, X


# --- Preflight check: run all three datasets and report ---
rows = []
for name in DATASET_REGISTRY:
    try:
        y, X = universal_loader(name)
        rows.append({
            'Dataset':    name,
            'N Rows':     len(y),
            'N Features': len(X.columns) - 1,
            'Base Rate':  f'{y.mean():.1%}',
            'Status':     'READY'
        })
    except Exception as e:
        rows.append({'Dataset': name, 'Status': f'ERROR: {str(e)[:60]}'})

display(pd.DataFrame(rows))

In [ ]:
# ============================================================
# CELL 4 — INTERACTIVE DASHBOARD
# ============================================================

# --- Widgets ---
dataset_dropdown = widgets.Dropdown(
    options=list(DATASET_REGISTRY.keys()),
    value='UCLA',
    description='Dataset:',
    style={'description_width': 'initial'}
)
threshold_slider = widgets.FloatSlider(
    value=0.5, min=0.05, max=0.95, step=0.05,
    description='Threshold:',
    readout_format='.2f',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
controls = widgets.VBox([dataset_dropdown, threshold_slider])


def update_dashboard(change):
    clear_output(wait=True)
    display(controls)

    ds_name = dataset_dropdown.value
    thresh  = threshold_slider.value

    try:
        # ---- Load & fit ----
        y, X = universal_loader(ds_name)
        y_f  = y.astype(float)
        X_f  = X.astype(float)

        model  = sm.Logit(y_f, X_f).fit_regularized(method='l1', alpha=0.1, disp=0)
        y_prob = model.predict(X_f)
        y_pred = (y_prob >= thresh).astype(int)

        acc  = accuracy_score(y, y_pred)
        cm   = confusion_matrix(y, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        # ---- Coefficients (exclude constant) ----
        coeff_df = (
            pd.DataFrame({'Feature': X_f.columns, 'Coeff': model.params})
            .query("Feature != 'const'")
            .sort_values('Coeff')
            .reset_index(drop=True)
        )

        # ================================================================
        # CHART LAYOUT — 16:9  (2 rows x 2 cols)
        #   [0,0] Sigmoid curve          [0,1] Factor weights
        #   [1,0] Separation histogram   [1,1] Confusion matrix heatmap
        # ================================================================
        fig = plt.figure(figsize=(16, 8))  # 16:8
        fig.suptitle(f'Logistic Regression Dashboard — {ds_name}  |  Threshold = {thresh:.2f}',
                     fontsize=15, fontweight='bold', y=1.01)

        gs = gridspec.GridSpec(2, 2, hspace=0.55, wspace=0.4)
        ax_sig  = fig.add_subplot(gs[0, 0])
        ax_coef = fig.add_subplot(gs[0, 1])
        ax_hist = fig.add_subplot(gs[1, 0])
        ax_cm   = fig.add_subplot(gs[1, 1])

        # ---- Panel A: True Sigmoid (linear score vs predicted probability) ----
        z_scores = X_f.values @ model.params.values
        sort_idx = np.argsort(z_scores)
        z_sorted = z_scores[sort_idx]
        p_sorted = y_prob.values[sort_idx]

        ax_sig.plot(z_sorted, p_sorted, color='steelblue', lw=2.5, label='P(outcome=1)')
        ax_sig.axhline(thresh, color='crimson', linestyle='--', lw=1.8,
                       label=f'Threshold = {thresh:.2f}')
        ax_sig.fill_between(z_sorted, p_sorted, alpha=0.12, color='steelblue')
        ax_sig.set_title('A. Sigmoid — P(Outcome=1) vs. Linear Score', fontweight='bold', fontsize=12)
        ax_sig.set_xlabel('Linear Score  z = β₀ + β₁x₁ + ... + βₙxₙ', fontsize=11)
        ax_sig.set_ylabel('P(Outcome = 1)', fontsize=11)
        ax_sig.set_ylim(-0.05, 1.05)
        ax_sig.legend(framealpha=0.7, fontsize=10)
        ax_sig.grid(axis='y', alpha=0.3)

        # Label ~10 evenly spaced points
        label_idx = np.linspace(0, len(z_sorted) - 1, 10, dtype=int)
        for i in label_idx:
            ax_sig.annotate(
                f'{p_sorted[i]:.2f}',
                xy=(z_sorted[i], p_sorted[i]),
                xytext=(0, 7), textcoords='offset points',
                ha='center', fontsize=9, color='steelblue'
            )

        # ---- Panel B: Factor Weights ----
        colors = ['#d73027' if c > 0 else '#4575b4' for c in coeff_df['Coeff']]
        bars = ax_coef.barh(coeff_df['Feature'], coeff_df['Coeff'],
                            color=colors, edgecolor='white', height=0.7)
        ax_coef.axvline(0, color='black', lw=0.9)
        ax_coef.set_title('B. Factor Weights (Log-Odds Coefficients)', fontweight='bold', fontsize=12)
        ax_coef.set_xlabel('Coefficient (log-odds)', fontsize=11)
        ax_coef.set_ylabel('Feature', fontsize=11)
        ax_coef.tick_params(labelsize=11)
        ax_coef.grid(axis='x', alpha=0.3)

        # Label each bar with its coefficient value
        x_range = coeff_df['Coeff'].abs().max()
        pad = x_range * 0.03
        for bar, val in zip(bars, coeff_df['Coeff']):
            x_pos = bar.get_width() + pad if val >= 0 else bar.get_width() - pad
            ha = 'left' if val >= 0 else 'right'
            ax_coef.text(x_pos, bar.get_y() + bar.get_height() / 2,
                         f'{val:.3f}', va='center', ha=ha, fontsize=11, fontweight='bold')

        # Widen xlim so labels don't get clipped
        xl = ax_coef.get_xlim()
        ax_coef.set_xlim(xl[0] - x_range * 0.18, xl[1] + x_range * 0.18)

        # ---- Panel C: Separation Histogram ----
        df_hist = pd.DataFrame({'Probability': y_prob, 'Actual': y.astype(int)})
        for outcome, color, label in [(0, '#d73027', 'Actual 0'), (1, '#1a9641', 'Actual 1')]:
            subset = df_hist.loc[df_hist['Actual'] == outcome, 'Probability']
            ax_hist.hist(subset, bins=20, density=True, alpha=0.55,
                        color=color, label=label, edgecolor='white')
        ax_hist.axvline(thresh, color='black', linestyle='--', lw=2,
                        label=f'Threshold = {thresh:.2f}')
        ax_hist.set_title('C. Predicted Probability by Actual Outcome', fontweight='bold', fontsize=12)
        ax_hist.set_xlabel('Predicted Probability of Outcome = 1', fontsize=11)
        ax_hist.set_ylabel('Density', fontsize=11)
        ax_hist.legend(framealpha=0.7, fontsize=10)
        ax_hist.grid(axis='y', alpha=0.3)

        # ---- Panel D: Confusion Matrix Heatmap ----
        cm_pct = cm.astype(float) / cm.sum()
        annot  = np.array([[f'{cm[i,j]}\n({cm_pct[i,j]:.1%})' for j in range(2)] for i in range(2)])
        sns.heatmap(
            cm, annot=annot, fmt='', ax=ax_cm,
            cmap='Blues', linewidths=1, linecolor='white',
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'],
            annot_kws={'size': 16, 'weight': 'bold'},
            cbar=False
        )
        ax_cm.set_title(
            f'D. Confusion Matrix  |  Accuracy = {acc:.1%}',
            fontweight='bold', fontsize=12
        )
        ax_cm.set_ylabel('Actual', fontsize=11)
        ax_cm.set_xlabel('Predicted', fontsize=11)
        ax_cm.tick_params(labelsize=11)

        # Accuracy stats under confusion matrix
        prec0 = tn / (tn + fn) if (tn + fn) > 0 else 0
        prec1 = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec0  = tn / (tn + fp) if (tn + fp) > 0 else 0
        rec1  = tp / (tp + fn) if (tp + fn) > 0 else 0
        stats_text = (f'Accuracy: {acc:.1%}     Base Rate: {y.mean():.1%}     N={len(y)}\n'
                      f'Precision  — Class 0: {prec0:.1%}   Class 1: {prec1:.1%}\n'
                      f'Recall     — Class 0: {rec0:.1%}   Class 1: {rec1:.1%}')
        ax_cm.text(0.5, -0.28, stats_text,
                   transform=ax_cm.transAxes,
                   ha='center', va='top', fontsize=10,
                   family='monospace',
                   bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f0f0', edgecolor='#cccccc'))

    except Exception as e:
        import traceback
        print(f'Dashboard error: {e}')
        traceback.print_exc()


# ---- Wire up widgets and launch ----
dataset_dropdown.observe(update_dashboard, names='value')
threshold_slider.observe(update_dashboard, names='value')
update_dashboard(None)